# Topic Modeling with BERTopic — Parameter Search & Evaluation (iGEM Teams)

Loads the pre-computed **iGEM Teams** embeddings, runs a grid search over
key UMAP/HDBSCAN parameters, evaluates each configuration with **C_v
coherence**, **topic diversity**, and **DBCV**, selects the best model,
optionally reassigns outliers, and saves the results to `assets/topic_models/`.

> **Recommended** — this is the notebook used in the associated publication.

In [ ]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 02-topic_model/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from aux.paths import MODELS_DIR, set_seed
from aux.topic_modeling import load_corpus, save_topic_outputs
from aux.evaluation import grid_search

set_seed()

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
EMBEDDINGS_FILE = "teams_embeddings.npy"
CORPUS_FILE     = "teams_corpus.txt"
ID_COL          = "UT"
PREFIX          = "teams"

PARAM_GRID = {
    "min_cluster_size": [8, 10, 15, 20],
    "umap_n_neighbors": [10, 15, 25],
    "umap_n_components": [5, 10],
}

# Reassign all iGEM noise documents to their nearest topic (see section 3).
REDUCE_OUTLIERS = True

## 1. Load embeddings and corpus

In [ ]:
embeddings, corpus = load_corpus(EMBEDDINGS_FILE, CORPUS_FILE)
docs = corpus["text"].tolist()
print(f"Teams: {embeddings.shape[0]:,} docs, {embeddings.shape[1]} dims")

## 2. Grid search

Fit and evaluate a BERTopic model for every parameter combination.

In [ ]:
results, best = grid_search(docs, embeddings, PARAM_GRID, label="Teams")
results

In [ ]:
print("Best configuration:")
for k in ["min_cluster_size", "n_neighbors", "n_components", "n_topics",
          "coherence_cv", "diversity", "dbcv", "outlier_frac"]:
    print(f"  {k:16s} = {best[k]}")

## 3. Reduce outliers (optional)

HDBSCAN labels documents that fall outside any dense cluster as topic **−1**
(noise). While that is acceptable for the SynBio literature (some papers may be
genuinely off-topic), every iGEM team project is by definition related to
synthetic biology — its text may simply be too short or idiosyncratic to land in
a cluster. BERTopic's `reduce_outliers` (strategy `"embeddings"`, threshold `0`)
reassigns **all** noise documents to their nearest topic by cosine similarity,
without retraining the model.

Controlled by `REDUCE_OUTLIERS` in the config above (enabled for teams,
disabled for papers).

In [ ]:
model = best["model"]
topics = list(best["topics"])

if REDUCE_OUTLIERS:
    before = sum(1 for t in topics if t == -1)
    topics = model.reduce_outliers(
        docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0,
    )
    model.update_topics(docs, topics=topics)
    after = sum(1 for t in topics if t == -1)
    print(f"Outliers: {before:,} → {after:,}")
else:
    print("Outlier reduction disabled — keeping HDBSCAN noise labels.")

## 4. Save best model, outputs, and grid-search results

In [ ]:
save_topic_outputs(model, corpus, topics, ID_COL, PREFIX)
results.to_csv(MODELS_DIR / f"{PREFIX}_grid_search.txt", sep="\t", index=False)

print(f"Saved → {MODELS_DIR}")
for f in sorted(MODELS_DIR.glob(f"{PREFIX}_*")):
    print(f"  {f.name}")